In [1]:
!pip install great-expectations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/HealthTrack/healthtrack_raw.csv")
print(df.shape)

Mounted at /content/drive
(300, 12)


In [5]:
import great_expectations as gx
print(gx.__version__)

1.18.1


In [6]:
import great_expectations as gx

context = gx.get_context()
data_source = context.data_sources.add_pandas("my_datasource")
data_asset = data_source.add_dataframe_asset("healthtrack")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
suite = context.suites.add(gx.ExpectationSuite(name="healthtrack_suite"))
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(name="healthtrack_val", data=batch_definition, suite=suite)
)
print("Setup done!")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpx2r_dij2' for ephemeral docs site


Setup done!


In [7]:
# 4. patient_id must not be null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="patient_id")
)

# 5. readmission_30d must not be null
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="readmission_30d")
)

# 6. readmission_30d must only be 0 or 1
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(column="readmission_30d", value_set=[0, 1])
)

# 7. total_bill_usd must be between 500 and 200000
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="total_bill_usd", min_value=500, max_value=200000)
)

# 8. department must be one of the 5 valid values
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="department",
        value_set=["Cardiology", "Oncology", "Orthopedics", "Neurology", "General Medicine"]
    )
)

print("5 expectations added!")

5 expectations added!


In [8]:
results = validation_definition.run(batch_parameters={"dataframe": df})
print(results)


Calculating Metrics:   0%|          | 0/37 [00:00<?, ?it/s]

{
  "success": false,
  "results": [
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "my_datasource-healthtrack",
          "column": "patient_id"
        },
        "meta": {},
        "id": "06caf938-9bb2-4ad3-ac0b-dcd7483d3c85",
        "severity": "critical"
      },
      "result": {
        "element_count": 300,
        "unexpected_count": 25,
        "unexpected_percent": 8.333333333333332,
        "partial_unexpected_list": [
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null,
          null
        ],
        "partial_unexpected_counts": [
          {
            "value": null,
            "count": 20
          }
        ]

In [9]:
results_dict = results.to_json_dict()

import json
html_content = f"""
<html><head><title>HealthTrack GX Report</title></head>
<body>
<h1>Great Expectations Validation Report</h1>
<h2>Success: {results_dict['success']}</h2>
<h3>Statistics</h3>
<p>Evaluated: {results_dict['statistics']['evaluated_expectations']}</p>
<p>Passed: {results_dict['statistics']['successful_expectations']}</p>
<p>Failed: {results_dict['statistics']['unsuccessful_expectations']}</p>
<pre>{json.dumps(results_dict, indent=2)}</pre>
</body></html>
"""

with open("/content/drive/MyDrive/HealthTrack/healthtrack_gx_report.html", "w") as f:
    f.write(html_content)

print("Report saved!")

Report saved!


## Validation Summary

Our automated data checks found that 25 patient records cannot be
identified due to missing ID information, and 30 records are incomplete
because their follow-up outcome was never recorded. Additionally, 15
billing records contain an invalid placeholder value instead of an actual
amount. These issues affect roughly 1 in 6 records and must be resolved
before the data can be safely used for patient care decisions.